<a href="https://colab.research.google.com/github/sarkarshreyas853-droid/Math-quiz-and-fun/blob/main/RAG_Chatbot_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Chatbot: run it in your browser

Run each cell **from top to bottom**: click the ▶ play button on the left of a cell and wait until it finishes (a green tick appears) before running the next one.

You need: text-based PDFs (not scans) and an Anthropic API key from console.anthropic.com (with a few dollars of credit).

## Step 1: Install and download the project (takes 2-3 minutes)

In [1]:
!pip install -q chromadb pypdf sentence-transformers anthropic
!git clone -q https://github.com/sarkarshreyas853-droid/RAG-BOT
%cd RAG-BOT
print("Done. Project downloaded.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69

## Step 2: Upload your PDFs
A file picker will appear. Choose 3 to 5 PDFs and wait for the upload to finish.

In [ ]:
from google.colab import files
import shutil, os
os.makedirs("data", exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join("data", name))
print("PDFs in data/:", [f for f in os.listdir("data") if f.endswith(".pdf")])

## Step 3: Enter your API key
A box will appear. Paste your key and press Enter. It is hidden and is not saved in this notebook.

In [ ]:
import os, getpass
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key: ")
print("Key set.")

## Step 4: Build the search index (the first run downloads a small model, about 1-2 minutes)
This builds two versions so you can compare chunk sizes on your CV.

In [ ]:
!python ingest.py --chunk 500 --overlap 50
!python ingest.py --chunk 250 --overlap 25

## Step 5: Ask a question
Change the question text and run the cell again as many times as you like.

In [ ]:
from ask import RAG
rag = RAG("chroma_500")

question = "What is the minimum attendance required?"   # <-- change this

answer, chunks = rag.answer(question)
print(answer)
print("\nSources:")
for c in chunks:
    print(" -", c["file"], "page", c["page"])

## Step 6: Write your test questions
Replace the examples with **20 real questions** about your PDFs. For each one write:
- `file`: the PDF's exact name
- `page`: the page number where the answer is
- `expected_answer`: a short correct answer

Lines starting with `EXAMPLE` are ignored, so delete those entries once you add yours.

In [ ]:
import json
QUESTIONS = [
    {"question": "EXAMPLE: What is the minimum attendance required?",
     "file": "your_file.pdf", "page": 1, "expected_answer": "75 percent."},
    # Add your real questions here, one block per question:
    # {"question": "...", "file": "your_file.pdf", "page": 5, "expected_answer": "..."},
]
json.dump(QUESTIONS, open("questions.json", "w"), indent=2)
print(len(QUESTIONS), "questions saved")

## Step 7: Run the evaluation
This gives you the numbers for your CV. Write the **Retrieval hit rate** and **Answer accuracy** for each version in your README table.

In [ ]:
print("===== Chunk size 500 =====")
!python eval.py --db chroma_500
print("\n===== Chunk size 250 =====")
!python eval.py --db chroma_250

## Done
Copy the two results into the table in your README on GitHub (Results section), and add a few sentences under "What I learned".

Colab forgets everything when you close it, so if you come back later, rerun Steps 1 to 4.